# FIT3182 Assignment 2 — Data Design & Streaming Application

This notebook covers:
- **Task 1** – MongoDB Data Model (collection design, schema, indexes, relationships)
- **Task 2** – Spark Structured Streaming application (ingestion → join → violation detection → MongoDB sink)
- **Task 3** – Code quality (comments, docstrings, markdown explanations)

---

## Task 1.1 — MongoDB Collection Design

Three core collections are used: `vehicles`, `cameras`, and `violations`.

### Collection: `vehicles`
Stores static registration and ownership data loaded from `vehicle.csv`.

| Field | Type | Description |
|---|---|---|
| `_id` | ObjectId | Auto-generated primary key |
| `car_plate` | String | Unique vehicle registration plate |
| `owner_name` | String | Registered owner full name |
| `owner_address` | String | Registered owner address |

**Sample document:**
```json
{
  "_id": ObjectId("..."),
  "car_plate": "ABC1234",
  "owner_name": "Ali bin Ahmad",
  "owner_address": "123 Jalan Utama, Kuala Lumpur"
}
```

**Indexes:** Unique index on `car_plate` — enforces no duplicates and supports fast O(log n) lookups from violation enrichment queries.

**Retention policy:** Permanent — vehicle registrations are a slowly changing reference dataset.

---

### Collection: `cameras`
Stores static camera metadata loaded from `camera.csv`.

| Field | Type | Description |
|---|---|---|
| `_id` | ObjectId | Auto-generated primary key |
| `camera_id` | Int | Unique camera identifier |
| `position_km` | Double | Distance along the monitored road (km) |
| `speed_limit` | Int | Instantaneous speed limit (km/h) at this camera |
| `segment_id` | String | Road segment identifier (links adjacent cameras) |

**Sample document:**
```json
{
  "_id": ObjectId("..."),
  "camera_id": 1,
  "position_km": 0.0,
  "speed_limit": 80,
  "segment_id": "SEG_01"
}
```

**Indexes:** Unique index on `camera_id`; compound index on `(segment_id, position_km)` to efficiently retrieve ordered cameras within a segment during average-speed calculations.

**Retention policy:** Permanent — camera positions change only when infrastructure changes.

---

### Collection: `violations`
Stores detected speed violations written by the streaming pipeline.

| Field | Type | Description |
|---|---|---|
| `_id` | ObjectId | Auto-generated primary key |
| `car_plate` | String | Offending vehicle plate |
| `violation_date` | String | Date of violation (YYYY-MM-DD) — used for daily merging |
| `violation_type` | String | `"instantaneous"` or `"average"` |
| `camera_id` | Int | Camera where violation was detected (end camera for average) |
| `speed_recorded` | Double | Measured speed (km/h) |
| `speed_limit` | Int | Applicable speed limit (km/h) |
| `event_time` | String | Full ISO-8601 timestamp of the event |
| `segment_start_camera` | Int | (Average violations only) Starting camera of the segment |
| `distance_km` | Double | (Average violations only) Segment length |
| `producer_source` | String | Which Kafka producer stream the event came from |

**Sample document (instantaneous):**
```json
{
  "_id": ObjectId("..."),
  "car_plate": "XYZ9999",
  "violation_date": "2024-01-15",
  "violation_type": "instantaneous",
  "camera_id": 2,
  "speed_recorded": 105.3,
  "speed_limit": 80,
  "event_time": "2024-01-15T08:23:45",
  "producer_source": "producer-A"
}
```

**Indexes:**
- Compound index on `(car_plate, violation_date)` — supports daily merging logic (upsert) and per-vehicle queries.
- Index on `violation_date` — supports time-range queries for reporting and visualisation.
- Index on `camera_id` — supports hotspot analysis per camera.

**Retention policy:** 90-day rolling window via a MongoDB TTL index on `event_time`. Older records are archived to cold storage (e.g., AWS S3) for audit purposes.

---

## Task 1.2 — Collection Relationships and Design Justification

```
vehicles (car_plate) ──────────── violations (car_plate)   [referenced]
cameras  (camera_id) ──────────── violations (camera_id)   [referenced]
```

**Embedding vs. referencing decision:**

All three collections use **referencing** (foreign-key style) rather than embedding.

| Trade-off | Justification |
|---|---|
| **Read pattern** | The streaming pipeline enriches events by joining against `cameras` once at ingestion. The violation document stores a snapshot of `camera_id` and `speed_limit` at detection time — so downstream reads of `violations` do not need to re-join `cameras`. |
| **Write pattern** | The streaming sink writes violations at high throughput. Embedding vehicle or camera sub-documents into every violation record would duplicate data unnecessarily and increase document size, reducing write throughput. |
| **Consistency** | Vehicle and camera data are static reference tables. Because they rarely change, referential integrity is easy to maintain without transactions. |
| **Duplication cost** | Storing `speed_limit` as a snapshot value in the violation is acceptable controlled duplication — it preserves the limit that was active at the time of the violation, which is important for enforcement. |


---

## Task 2 — Streaming Application

### Architecture Overview

```
camera_event_A.csv ──► Kafka Producer A ──► topic: camera-events-A ──┐
camera_event_B.csv ──► Kafka Producer B ──► topic: camera-events-B ──┤── Spark Structured Streaming
camera_event_C.csv ──► Kafka Producer C ──► topic: camera-events-C ──┘         │
                                                                                │
                                         ┌──────────────────────────────────────┘
                                         ▼
                              Enrich with camera.csv (static join)
                                         │
                         ┌───────────────┴───────────────┐
                         ▼                               ▼
               Instantaneous check            Stream-stream join
               speed_reading > limit          (car_plate + segment + time window)
                         │                               │
                         │                   Average speed calculation
                         │                   avg_speed > end_camera limit
                         └───────────────┬───────────────┘
                                         ▼
                              foreachBatch → upsert to MongoDB
                                    (violations collection)
```

### Key Parameters

| Parameter | Value | Rationale |
|---|---|---|
| Watermark | 10 minutes | Allows late events up to 10 min; balances lateness tolerance vs. state size |
| Avg-speed time window | 30 minutes | Maximum plausible travel time across a 2-camera segment at walking speed |
| Spark shuffle partitions | 2 | Local development; increase for production cluster |
| Kafka `startingOffsets` | `latest` | Process only new events; use `earliest` for full replay |
| MongoDB bulk write size | foreachBatch micro-batch | Amortises connection overhead; one bulk per batch |


### 2.1.1 — Spark Setup and Configuration

In [ ]:
import os
from pathlib import Path
from datetime import datetime

from pymongo import MongoClient, UpdateOne
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, expr, from_json, to_timestamp, abs as spark_abs,
    unix_timestamp, lit, to_date
)
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType
)

# ---------------------------------------------------------------------------
# Configuration — change HOST_IP to match your Kafka broker address
# ---------------------------------------------------------------------------
HOST_IP = "192.168.64.1"
KAFKA_BOOTSTRAP_SERVERS = f"{HOST_IP}:9092"

SPARK_PACKAGES = (
    "org.apache.spark:spark-streaming-kafka-0-10_2.12:3.5.5,"
    "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.5"
)

os.environ["PYSPARK_SUBMIT_ARGS"] = f"--packages {SPARK_PACKAGES} pyspark-shell"

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("FIT3182-A2-AWAS-Streaming")
    # Reduce shuffle partitions for local development (default 200 is too high)
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.streaming.stopGracefullyOnShutdown", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")
print("Spark session started successfully.")


### 2.1.1 — Kafka Stream Ingestion

Each of the three camera event CSVs is published by a separate Kafka producer into its own topic. This cell reads all three topics as Spark structured streams, parses the JSON payloads, and applies a watermark to bound streaming state.

In [ ]:
# ---------------------------------------------------------------------------
# JSON schema matching the producer payload
# Fields: event_id, batch_id, car_plate, camera_id, timestamp, speed_reading
# ---------------------------------------------------------------------------
event_schema = StructType([
    StructField("event_id",     StringType(),  nullable=False),
    StructField("batch_id",     IntegerType(), nullable=False),
    StructField("car_plate",    StringType(),  nullable=False),
    StructField("camera_id",    IntegerType(), nullable=False),
    StructField("timestamp",    StringType(),  nullable=False),
    StructField("speed_reading", DoubleType(), nullable=False),
])


def read_camera_stream(topic: str, producer_label: str):
    """Read a Kafka topic and return a watermarked streaming DataFrame.

    Args:
        topic: Kafka topic name to subscribe to.
        producer_label: Human-readable producer identifier added as metadata.

    Returns:
        A Spark streaming DataFrame with columns:
        event_id, batch_id, car_plate, camera_id, event_time (timestamp),
        speed_reading, producer_source.
    """
    return (
        spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS)
        .option("subscribe", topic)
        .option("startingOffsets", "latest")
        .option("failOnDataLoss", "false")
        .load()
        # Kafka value arrives as bytes — cast to string first
        .selectExpr("CAST(value AS STRING) AS json_value")
        # Parse JSON string into struct columns using our schema
        .select(from_json(col("json_value"), event_schema).alias("data"))
        .select("data.*")
        # Convert string timestamp to proper Spark timestamp type
        .withColumn("event_time", to_timestamp(col("timestamp")))
        # Tag each event with its originating producer for traceability
        .withColumn("producer_source", lit(producer_label))
        # Watermark: tolerate events arriving up to 10 minutes late.
        # Spark uses this to decide when it is safe to expire join state.
        .withWatermark("event_time", "10 minutes")
    )


# Read all three producer topics
stream_a = read_camera_stream("camera-events-A", "producer-A")
stream_b = read_camera_stream("camera-events-B", "producer-B")
stream_c = read_camera_stream("camera-events-C", "producer-C")

print("Kafka streams created for topics: camera-events-A, camera-events-B, camera-events-C")


### Static Enrichment — Join Camera Metadata

Camera metadata (position and speed limit per camera) is a small, static dataset loaded from `camera.csv`. It is joined as a static DataFrame against each stream to enrich events with `speed_limit` and `position_km`. This is a stream-static join (no watermark needed on the static side).

In [ ]:
# ---------------------------------------------------------------------------
# Load camera reference data from CSV (static — not a stream)
# ---------------------------------------------------------------------------
camera_df = (
    spark.read
    .csv(str(Path("../data/camera.csv")), header=True, inferSchema=True)
    # Keep only the columns needed for enrichment and violation detection
    .select("camera_id", "speed_limit", "position_km", "segment_id")
)

camera_df.show()
print(f"Camera metadata loaded: {camera_df.count()} cameras")


def enrich_with_camera(stream):
    """Join a camera event stream with static camera metadata.

    Args:
        stream: Spark streaming DataFrame with a camera_id column.

    Returns:
        Streaming DataFrame enriched with speed_limit, position_km, segment_id.
    """
    return stream.join(camera_df, on="camera_id", how="inner")


enriched_a = enrich_with_camera(stream_a)
enriched_b = enrich_with_camera(stream_b)
enriched_c = enrich_with_camera(stream_c)

print("Streams enriched with camera metadata.")


### 2.1.2 — Streaming Join Logic

#### Instantaneous Speed Violations

Instantaneous violations are detected independently on each stream — no cross-stream join is needed. A simple filter checks whether `speed_reading > speed_limit`.

#### Average Speed Violations

Average speed violations require matching a vehicle's **entry event** (earlier camera) with its **exit event** (later camera on the same road segment). This is a **stream-stream join**.

Following the teaching example (`C_spark_structured_streaming_join.ipynb`), each stream is aliased and joined using an `expr()` condition that includes:
1. Same `car_plate` (the vehicle must match)
2. Same `segment_id` (cameras must be on the same monitored segment)
3. The entry camera must have a lower `position_km` than the exit camera
4. The exit event must occur within 30 minutes of the entry event (bounded time window)

Watermarks on both sides are required for Spark to safely evict expired join state.


In [ ]:
# ---------------------------------------------------------------------------
# Task 2.1.4 — Instantaneous violation detection
# Filter events where the recorded speed exceeds the camera speed limit
# ---------------------------------------------------------------------------

def detect_instant_violations(enriched_stream):
    """Detect instantaneous speed violations on a single stream.

    A violation is raised when speed_reading exceeds the speed_limit
    of the camera that recorded the event.

    Args:
        enriched_stream: Stream enriched with camera metadata.

    Returns:
        Filtered streaming DataFrame containing only violating rows,
        with a violation_type column set to 'instantaneous'.
    """
    return (
        enriched_stream
        .filter(col("speed_reading") > col("speed_limit"))
        .withColumn("violation_type", lit("instantaneous"))
        .withColumn("violation_date", to_date(col("event_time")))
        .select(
            "car_plate",
            "violation_date",
            "violation_type",
            "camera_id",
            col("speed_reading").alias("speed_recorded"),
            "speed_limit",
            col("event_time").cast("string").alias("event_time"),
            "producer_source",
        )
    )


instant_violations_a = detect_instant_violations(enriched_a)
instant_violations_b = detect_instant_violations(enriched_b)
instant_violations_c = detect_instant_violations(enriched_c)

# Combine instantaneous violations from all three producers
instant_violations = (
    instant_violations_a
    .union(instant_violations_b)
    .union(instant_violations_c)
)

print("Instantaneous violation detection logic defined.")


In [ ]:
# ---------------------------------------------------------------------------
# Task 2.1.2 — Stream-stream join for average speed violations
#
# Pattern mirrors the teaching example (C_spark_structured_streaming_join):
#   - Both sides have a watermark applied before the join
#   - Join condition uses expr() with timestamp bounds
#   - Inner join — only matched (entry, exit) pairs produce output
#
# The join is performed across all three enriched streams combined into
# a single "all cameras" stream so that entry and exit events from
# different producers can be matched.
# ---------------------------------------------------------------------------

# Combine all three enriched streams into one for the segment join.
# Watermark is already applied per-stream inside read_camera_stream().
all_events = enriched_a.union(enriched_b).union(enriched_c)

# Alias the same combined stream as entry (start of segment) and
# exit (end of segment) sides of the join
entry_events = all_events.alias("entry")
exit_events  = all_events.alias("exit")

# Stream-stream join:
#   - Same vehicle (car_plate)
#   - Same road segment (segment_id)
#   - Exit camera is further along the road than entry camera
#   - Exit event occurs after the entry event, within a 30-minute window
#     (30 min is the upper bound for a ~2 km segment at very low speed)
segment_joined = (
    entry_events.join(
        exit_events,
        expr("""
            entry.car_plate  = exit.car_plate
            AND entry.segment_id = exit.segment_id
            AND entry.position_km < exit.position_km
            AND exit.event_time  > entry.event_time
            AND exit.event_time <= entry.event_time + interval 30 minutes
        """),
        "inner"
    )
    .select(
        col("entry.car_plate").alias("car_plate"),
        col("entry.camera_id").alias("segment_start_camera"),
        col("exit.camera_id").alias("camera_id"),
        col("entry.segment_id").alias("segment_id"),
        col("entry.event_time").alias("entry_time"),
        col("exit.event_time").alias("exit_time"),
        col("entry.position_km").alias("entry_position_km"),
        col("exit.position_km").alias("exit_position_km"),
        col("exit.speed_limit").alias("speed_limit"),   # rule: compare against END camera limit
        col("exit.producer_source").alias("producer_source"),
    )
)

print("Stream-stream segment join defined.")


In [ ]:
# ---------------------------------------------------------------------------
# Task 2.1.4 — Average speed calculation and violation detection
# ---------------------------------------------------------------------------

def compute_average_speed(joined_df):
    """Compute average speed across a road segment and flag violations.

    Average speed = distance (km) / travel time (hours).
    A violation is raised when average speed exceeds the end-camera speed limit.

    Args:
        joined_df: DataFrame from the segment stream-stream join.

    Returns:
        DataFrame containing only average-speed violating rows.
    """
    return (
        joined_df
        .withColumn(
            "distance_km",
            spark_abs(col("exit_position_km") - col("entry_position_km"))
        )
        .withColumn(
            "travel_time_hours",
            (unix_timestamp(col("exit_time")) - unix_timestamp(col("entry_time"))) / 3600.0
        )
        .withColumn(
            "average_speed",
            col("distance_km") / col("travel_time_hours")
        )
        # Only flag if average speed exceeds the ending camera's speed limit
        .filter(col("average_speed") > col("speed_limit"))
        .withColumn("violation_type", lit("average"))
        .withColumn("violation_date", to_date(col("exit_time")))
        .select(
            "car_plate",
            "violation_date",
            "violation_type",
            "camera_id",                             # exit camera
            col("average_speed").alias("speed_recorded"),
            "speed_limit",
            col("exit_time").cast("string").alias("event_time"),
            "segment_start_camera",
            "distance_km",
            "producer_source",
        )
    )


average_violations = compute_average_speed(segment_joined)

print("Average speed violation detection logic defined.")


### 2.1.3 — MongoDB Sink Integration

Violations are persisted to MongoDB using `foreachBatch`, which gives us access to the full batch DataFrame so we can use pymongo's `bulk_write` for efficiency.

**Idempotency / daily merging (Task 2.1.4):** The assignment requires that multiple violations for the same `(car_plate, violation_date)` are **merged** into a single daily record. This is achieved with a MongoDB `UpdateOne` upsert keyed on `(car_plate, violation_date, violation_type, camera_id)`. If the record already exists, `$push` appends the new speed reading to an `incidents` array rather than creating a duplicate document.

**Retry handling:** `pymongo`'s `MongoClient` automatically retries retryable writes (e.g., primary failover). An explicit try/except catches non-retryable errors and logs them without crashing the stream.


In [ ]:
# ---------------------------------------------------------------------------
# MongoDB connection settings
# ---------------------------------------------------------------------------
MONGO_URI = "mongodb://localhost:27017/"
MONGO_DB  = "awas_traffic"
VIOLATIONS_COLLECTION = "violations"


def get_mongo_collection():
    """Return a handle to the violations MongoDB collection.

    Creates a new MongoClient per executor call; pymongo is not
    serialisable so it cannot be shared across foreachBatch calls.

    Returns:
        pymongo Collection object.
    """
    client = MongoClient(MONGO_URI)
    return client[MONGO_DB][VIOLATIONS_COLLECTION], client


def write_violations_to_mongo(batch_df, batch_id: int) -> None:
    """foreachBatch sink: upsert violation records into MongoDB.

    Each row is upserted using (car_plate, violation_date, violation_type,
    camera_id) as the compound key. The recorded speed is pushed into an
    'incidents' array so that multiple violations by the same vehicle on the
    same day at the same camera are merged into one document.

    Args:
        batch_df: Spark DataFrame for the current micro-batch.
        batch_id: Spark-assigned integer batch identifier (for logging).
    """
    rows = batch_df.collect()

    if not rows:
        print(f"[Batch {batch_id}] No violations to write.")
        return

    operations = []
    for row in rows:
        filter_key = {
            "car_plate":      row["car_plate"],
            "violation_date": str(row["violation_date"]),
            "violation_type": row["violation_type"],
            "camera_id":      row["camera_id"],
        }
        update_doc = {
            "$setOnInsert": {
                "speed_limit":    row["speed_limit"],
                "producer_source": row["producer_source"],
            },
            "$push": {
                "incidents": {
                    "speed_recorded": row["speed_recorded"],
                    "event_time":     row["event_time"],
                }
            }
        }
        operations.append(UpdateOne(filter_key, update_doc, upsert=True))

    collection, client = get_mongo_collection()
    try:
        result = collection.bulk_write(operations, ordered=False)
        print(
            f"[Batch {batch_id}] Written {len(operations)} violation(s) to MongoDB "
            f"(upserted: {result.upserted_count}, modified: {result.modified_count})."
        )
    except Exception as exc:
        # Log and continue — do not crash the streaming query on a write error
        print(f"[Batch {batch_id}] MongoDB write error: {exc}")
    finally:
        client.close()


print("MongoDB sink function defined.")


### 2.1.1 / 2.1.2 — Start Streaming Queries

Three queries are started:
1. **Diagnostic query** — prints combined raw stream to console (helps verify ingestion is working)
2. **Instantaneous violations query** — writes instant violations to MongoDB
3. **Average speed violations query** — writes average violations to MongoDB

All queries use `checkpointLocation` so Spark can recover offset state after a restart.

In [ ]:
import os

os.makedirs("checkpoints/diagnostic",    exist_ok=True)
os.makedirs("checkpoints/instant_mongo", exist_ok=True)
os.makedirs("checkpoints/average_mongo", exist_ok=True)

# ---------------------------------------------------------------------------
# Query 1: Diagnostic — print raw combined stream to console
# Useful for verifying Kafka ingestion is working before inspecting MongoDB
# ---------------------------------------------------------------------------
diagnostic_query = (
    all_events.writeStream
    .foreachBatch(lambda df, bid: (
        print(f"\n[Diagnostic Batch {bid}] {df.count()} event(s) received."),
        df.show(5, truncate=False)
    ))
    .outputMode("append")
    .option("checkpointLocation", "checkpoints/diagnostic")
    .start()
)

# ---------------------------------------------------------------------------
# Query 2: Instantaneous violations → MongoDB
# ---------------------------------------------------------------------------
instant_query = (
    instant_violations.writeStream
    .foreachBatch(write_violations_to_mongo)
    .outputMode("append")
    .option("checkpointLocation", "checkpoints/instant_mongo")
    .start()
)

# ---------------------------------------------------------------------------
# Query 3: Average speed violations → MongoDB
# ---------------------------------------------------------------------------
average_query = (
    average_violations.writeStream
    .foreachBatch(write_violations_to_mongo)
    .outputMode("append")
    .option("checkpointLocation", "checkpoints/average_mongo")
    .start()
)

print("All streaming queries started.")
print("Waiting for data from Kafka producers...")

# Block until any query terminates (or interrupt manually to stop)
spark.streams.awaitAnyTermination()


### Stopping the Streaming Queries

Run this cell to gracefully stop all active queries after the producers have finished.

In [ ]:
# Gracefully stop all streaming queries
for q in spark.streams.active:
    q.stop()
    print(f"Stopped query: {q.name or q.id}")

print("All streaming queries stopped.")
